# HW2: LLM Alignment

**CMSC 25750: Large Language Models** | Instructor: Chenhao Tan

This notebook is your submission file. Run all cells in order after compiling each section, then write your answers in the designated markdown cells.

**Before you start**: Make sure you have run `compile.py` for the relevant section(s) so that the `cache/` directory is populated.

---
**Sections**
- [Section 1: SFT](#section1)
- [Section 2: Instruction Tuning](#section2)
- [Section 3: RLHF via GRPO](#section3)

In [ ]:
# ── Setup ─────────────────────────────────────────────────────────────────────
import sys, os
sys.path.insert(0, os.path.dirname(os.path.abspath('__file__')))

import json
import torch
import matplotlib.pyplot as plt
import matplotlib
from pathlib import Path

matplotlib.rcParams['figure.dpi'] = 120
matplotlib.rcParams['font.size']  = 11

device    = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
CACHE_DIR = Path('cache')

print(f'Device    : {device}')
print(f'Cache dir : {CACHE_DIR.resolve()}')
print(f'Cache contents: {[f.name for f in CACHE_DIR.iterdir()] if CACHE_DIR.exists() else "(empty)"}')

---
<a id='section1'></a>
## Section 1: Supervised Fine-Tuning (SFT)

In [ ]:
# Autograder 
# Run this cell after implementing src/sft.py
!pytest tests/test_sft.py -v 2>&1 | tail -30

In [ ]:
# Load SFT results from cache 
SFT_CACHE = CACHE_DIR / 'sft_results.json'
assert SFT_CACHE.exists(), (
    "cache/sft_results.json not found. "
    "Run: python compile.py section1"
)
with open(SFT_CACHE) as f:
    sft = json.load(f)

print(f"Model      : {sft['model_name']}")
print(f"Train losses per epoch: {[f'{x:.4f}' for x in sft.get('epoch_losses', sft.get('train_losses', []))]}")
print(f"Val ROUGE-1 per epoch : {[f'{x:.4f}' for x in sft.get('epoch_rouge', sft.get('val_rouge', []))]}")
print(f"Logged training steps : {len(sft.get('train_steps', []))}")

In [ ]:
# Plot training loss and ROUGE
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

# Use detailed step-wise losses if available, otherwise fall back to epoch losses
if 'train_steps' in sft and len(sft['train_steps']) > 0:
    axes[0].plot(sft['train_steps'], sft['train_losses'], '-', color='steelblue', alpha=0.7)
    axes[0].set_xlabel('Training Steps')
else:
    epochs = list(range(1, len(sft.get('epoch_losses', sft.get('train_losses', []))) + 1))
    axes[0].plot(epochs, sft.get('epoch_losses', sft.get('train_losses', [])), 'o-', color='steelblue')
    axes[0].set_xlabel('Epoch')
    axes[0].set_xticks(epochs)

axes[0].set_title('SFT — Training Loss')
axes[0].set_ylabel('Cross-Entropy Loss')
axes[0].grid(alpha=0.3)

epochs = list(range(1, len(sft.get('epoch_rouge', sft.get('val_rouge', []))) + 1))
axes[1].plot(epochs, sft.get('epoch_rouge', sft.get('val_rouge', [])), 's-', color='darkorange')
axes[1].set_title('SFT — Validation ROUGE-1')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('ROUGE-1 F1')
axes[1].set_xticks(epochs)
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()
print(f"Final ROUGE-1: {sft.get('epoch_rouge', sft.get('val_rouge', []))[-1]:.4f}")

In [ ]:
# Qualitative examples 
print("=" * 70)
print("SFT QUALITATIVE EXAMPLES")
print("=" * 70)
for i, ex in enumerate(sft['examples'][:3]):
    print(f"\n--- Example {i+1} ---")
    print(f"Article (first 300 chars):\n  {ex['article'][:300]}...")
    print(f"Reference:\n  {ex['reference']}")
    print(f"SFT Prediction:\n  {ex['prediction']}")

### 1.4(a) — Training Loss Curve (4 pts)

> Inspect the training loss curve. Does the loss decrease across epochs? What does a decreasing loss tell us about what the model is learning?

**YOUR ANSWER:**

*(2–3 sentences.)*

### 1.4(b) — Comparison with First-Sentence Baseline (4 pts)

The cell below computes a first-sentence baseline.

In [ ]:
# First-sentence baseline
import evaluate
rouge_metric = evaluate.load('rouge')

from src.sft import get_data
_, val_data = get_data(num_train=5000, num_val=500)

n_eval = 100
first_sent_preds, refs = [], []
for item in list(val_data)[:n_eval]:
    first_sent_preds.append(item['article'].split('.')[0].strip() + '.')
    refs.append(item['highlights'])

baseline_score = rouge_metric.compute(
    predictions=first_sent_preds, references=refs
)['rouge1']
print(f"First-sentence baseline ROUGE-1 : {baseline_score:.4f}")
print(f"SFT model ROUGE-1               : {sft.get('epoch_rouge', sft.get('val_rouge', []))[-1]:.4f}")
print(f"Improvement                     : {sft.get('epoch_rouge', sft.get('val_rouge', []))[-1] - baseline_score:+.4f}")

> Compare your SFT ROUGE-1 score against the first-sentence baseline. What does the comparison tell you about the difficulty of the task?

**YOUR ANSWER:**

*(2–3 sentences.)*

### 1.4(c) — Why Mask Article Tokens? (4 pts)

> In the current implementation, the loss is computed on all tokens (article + separator + summary). Why might computing loss on the article tokens be suboptimal for an alignment objective? What signal are those tokens providing to the optimizer?

**YOUR ANSWER:**

*(2–3 sentences.)*

---
<a id='section2'></a>
## Section 2: Instruction Tuning

In [ ]:
# Autograder 
!pytest tests/test_instruction_tuning.py -v 2>&1 | tail -30

In [ ]:
# Load IT results from cache 
IT_CACHE = CACHE_DIR / 'it_results.json'
assert IT_CACHE.exists(), (
    "cache/it_results.json not found. "
    "Run: python compile.py section2"
)
with open(IT_CACHE) as f:
    it = json.load(f)

print(f"Model      : {it['model_name']}")
print(f"Train losses per epoch: {[f'{x:.4f}' for x in it.get('epoch_losses', it.get('train_losses', []))]}")
print(f"Val ROUGE-1 per epoch : {[f'{x:.4f}' for x in it.get('epoch_rouge', it.get('val_rouge', []))]}")
print(f"Logged training steps : {len(it.get('train_steps', []))}")

In [ ]:
# Compare SFT vs IT training curves 
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

# Plot detailed training losses if available
if 'train_steps' in sft and len(sft['train_steps']) > 0:
    axes[0].plot(sft['train_steps'], sft['train_losses'], '-', color='steelblue', alpha=0.6, label='SFT')
else:
    epochs_sft = list(range(1, len(sft.get('epoch_losses', sft.get('train_losses', []))) + 1))
    axes[0].plot(epochs_sft, sft.get('epoch_losses', sft.get('train_losses', [])), 'o-', color='steelblue', label='SFT')

if 'train_steps' in it and len(it['train_steps']) > 0:
    axes[0].plot(it['train_steps'], it['train_losses'], '-', color='darkorange', alpha=0.6, label='IT')
else:
    epochs_it  = list(range(1, len(it.get('epoch_losses', it.get('train_losses', []))) + 1))
    axes[0].plot(epochs_it,  it.get('epoch_losses', it.get('train_losses', [])),  's-', color='darkorange', label='IT')

axes[0].set_title('Training Loss: SFT vs IT')
axes[0].set_xlabel('Training Steps' if 'train_steps' in sft else 'Epoch')
axes[0].set_ylabel('Cross-Entropy Loss')
axes[0].legend()
axes[0].grid(alpha=0.3)

# Epoch-based ROUGE comparison
epochs_sft = list(range(1, len(sft.get('epoch_rouge', sft.get('val_rouge', []))) + 1))
epochs_it  = list(range(1, len(it.get('epoch_rouge', it.get('val_rouge', []))) + 1))
axes[1].plot(epochs_sft, sft.get('epoch_rouge', sft.get('val_rouge', [])), 'o-', color='steelblue',  label='SFT')
axes[1].plot(epochs_it,  it.get('epoch_rouge', it.get('val_rouge', [])),  's-', color='darkorange', label='IT')
axes[1].set_title('Validation ROUGE-1: SFT vs IT')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('ROUGE-1 F1')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

print(f"SFT final ROUGE-1 : {sft.get('epoch_rouge', sft.get('val_rouge', []))[-1]:.4f}")
print(f"IT  final ROUGE-1 : {it.get('epoch_rouge', it.get('val_rouge', []))[-1]:.4f}")

In [ ]:
# Qualitative comparison 
print("=" * 70)
print("INSTRUCTION TUNING — QUALITATIVE EXAMPLES")
print("=" * 70)
for i in range(min(3, len(it['examples']))):
    ex_sft = sft['examples'][i]
    ex_it  = it['examples'][i]
    print(f"\n--- Example {i+1} ---")
    print(f"Article (first 300 chars):\n  {ex_it['article'][:300]}...")
    print(f"Reference       :\n  {ex_it['reference']}")
    print(f"SFT Prediction  :\n  {ex_sft['prediction']}")
    print(f"IT  Prediction  :\n  {ex_it['prediction']}")

In [ ]:
# Verify your format_instruction function 
from src.instruction_tuning import format_instruction, INSTRUCTION_TEMPLATE

sample_article = "Scientists have discovered a new species of deep-sea fish near the Mariana Trench."
sample_summary = "New deep-sea fish species found near Mariana Trench."

print("Prompt only (for generation):")
print("-" * 50)
print(format_instruction(sample_article))
print()
print("Full training sequence (with summary):")
print("-" * 50)
print(format_instruction(sample_article, sample_summary))

### 2.3(a) — SFT vs IT Comparison (5 pts)

> Compare the ROUGE-1 scores of the SFT model and the IT model. Also inspect the qualitative examples above. In what qualitative ways do the summaries differ? Which model produces more instruction-following outputs?

**YOUR ANSWER:**

*(3–4 sentences.)*

### 2.3(b) — Effect of Removing Response Masking (5 pts)

> Suppose you remove response masking and compute the loss on all tokens (as in Section 1), but keep the instruction template. Would you expect the model to learn instruction following? Why or why not?

**YOUR ANSWER:**

*(2–3 sentences.)*

### 2.4(a) — Chat Templates (5 pts)

> Modern instruction-tuned models like Llama-3.1-Instruct use a structured chat template. Why do models need a standardized chat template? What problems could arise if users prompt an instruction-tuned model without its expected template?

**YOUR ANSWER:**

*(3–4 sentences.)*

### 2.4(b) — Data Quantity vs. Quality (7 pts)

> Suppose you collected 1M random (instruction, response) pairs scraped from the internet with no quality filtering. Would training on more data necessarily improve instruction following? What is the key insight from LIMA (Zhou et al., 2023) about data quantity vs. quality?

**YOUR ANSWER:**

*(3–4 sentences.)*

---
<a id='section3'></a>
## Section 3: RLHF via GRPO

In [ ]:
# Autograder 
!pytest tests/test_rlhf.py -v 2>&1 | tail -30

In [ ]:
# Verify GRPO building blocks (Problems 3.1–3.3) 
import torch
from src.rlhf import (
    compute_rouge_reward,
    compute_group_normalized_rewards,
    compute_grpo_clip_loss,
)

# Problem 3.1 — reward function
completions   = ["the cat sat on the mat", "hello world"]
ground_truths = ["the cat sat on the mat", "foo bar baz"]
rewards = compute_rouge_reward(completions, ground_truths)
print(f"ROUGE rewards: {rewards}")
print(f"  Exact match should be ~1.0 : {rewards[0]:.4f}")
print(f"  Mismatch should be low     : {rewards[1]:.4f}")

# Problem 3.2 — group normalization
sample_rewards = [0.2, 0.8, 0.3, 0.7, 0.5, 0.5]
advantages = compute_group_normalized_rewards(sample_rewards, group_size=2)
print(f"\nAdvantages (group_size=2): {advantages}")
print(f"  Group 1 mean (should be ~0): {advantages[:2].mean():.4f}")
print(f"  Group 2 mean (should be ~0): {advantages[2:4].mean():.4f}")

# Problem 3.3 — clipped surrogate loss
log_p     = torch.zeros(4)
old_log_p = torch.zeros(4)
adv       = torch.tensor([1.0, -1.0, 1.0, -1.0])
loss = compute_grpo_clip_loss(log_p, old_log_p, adv)
print(f"\nGRPO clip loss with ratio=1, mean(adv)=0 (should be 0.0): {loss.item():.4f}")

In [ ]:
# Load RLHF results from cache (produced by compile.py section3) 
# compile.py calls your run_grpo_training() implementation and saves the results.
# This cell simply loads those results for visualization.
RLHF_CACHE = CACHE_DIR / 'rlhf_results.json'
assert RLHF_CACHE.exists(), (
    "cache/rlhf_results.json not found. "
    "Run: python compile.py section3"
)
with open(RLHF_CACHE) as f:
    rlhf = json.load(f)

print(f"Starting checkpoint : {rlhf['model_name']}")
print(f"Training steps      : {len(rlhf['train_steps'])}")
print(f"Final ROUGE-1       : {rlhf['val_rouge']:.4f}")

In [ ]:
# Plot GRPO training curves
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

if rlhf.get('train_steps') and rlhf.get('train_losses'):
    axes[0].plot(rlhf['train_steps'], rlhf['train_losses'], color='steelblue')
    axes[0].set_title('GRPO — Training Loss')
    axes[0].set_xlabel('Step')
    axes[0].set_ylabel('Loss')
    axes[0].grid(alpha=0.3)

if rlhf.get('reward_steps') and rlhf.get('rewards'):
    axes[1].plot(rlhf['reward_steps'], rlhf['rewards'], color='darkorange')
    axes[1].set_title('GRPO — Mean Training Reward (ROUGE-1)')
    axes[1].set_xlabel('Step')
    axes[1].set_ylabel('Reward')
    axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Final three-way comparison table 
cmp     = rlhf.get('comparison', {})
sft_r1  = cmp.get('sft_rouge1',  sft.get('epoch_rouge', sft.get('val_rouge', []))[-1])
it_r1   = cmp.get('it_rouge1',   it.get('epoch_rouge', it.get('val_rouge', []))[-1])
rlhf_r1 = cmp.get('rlhf_rouge1', rlhf['val_rouge'])

print("\n" + "=" * 45)
print(f"{'Method':<25} {'Val ROUGE-1 F1':>15}")
print("-" * 45)
print(f"{'SFT':<25} {sft_r1:>15.4f}")
print(f"{'Instruction Tuning':<25} {it_r1:>15.4f}")
print(f"{'RLHF (GRPO)':<25} {rlhf_r1:>15.4f}")
print("=" * 45)

### 3.5(a) — Final ROUGE Comparison (4 pts)

> Report the final ROUGE-1 F1 scores for all three methods in the table above. Which method achieves the highest ROUGE-1 score? Does this match your expectation? Provide one possible explanation for the ordering.

**YOUR ANSWER:**

*(2–3 sentences.)*

### 3.5(b) — Reward Curve Interpretation (3 pts)

> Look at the GRPO reward curve plotted above. Does the mean reward increase during training? What would a flat or decreasing reward curve indicate?

**YOUR ANSWER:**

*(2–3 sentences.)*

### 3.5(c) — Reward Hacking (3 pts)

> ROUGE measures n-gram overlap with a reference summary. Describe two failure modes where a model could achieve a high ROUGE score while producing a *poor* summary. This is an example of *reward hacking* — the model optimising the proxy metric rather than the true objective.

**YOUR ANSWER:**

1. *(Failure mode 1: one sentence)*

2. *(Failure mode 2: one sentence)*